In [2]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ROOT_USER = os.environ["MINIO_ROOT_USER"]
MINIO_ROOT_PASSWORD = os.environ["MINIO_ROOT_PASSWORD"]
BUCKET = "bronze-veloz"

builder = (
    SparkSession.builder.appName("delta-lake-tutorial")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ROOT_USER)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_ROOT_PASSWORD)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)
spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    ],
).getOrCreate()

/home/airflow/.local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found


:: loading settings :: url = jar:file:/home/airflow/.local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/airflow/.ivy2/cache
The jars for the packages stored in: /home/airflow/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d71f08fd-fe76-4dbc-8a61-be142befb7fb;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (269ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar

### Orders

In [12]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, FloatType, DateType
)
from pyspark.sql.functions import lit, to_date

orders_schema = StructType([
    StructField("order_id",     StringType(),    False),
    StructField("store_id",     StringType(),    False),
    StructField("rider_id",     StringType(),    True),
    StructField("status",       StringType(),    False),
    StructField("created_at",   TimestampType(), False),
    StructField("assigned_at",  TimestampType(), True),
    StructField("picked_up_at", TimestampType(), True),
    StructField("delivered_at", TimestampType(), True),
    StructField("order_total",  FloatType(),     False),
    StructField("updated_at",   TimestampType(), False),
    StructField("extract_date", DateType(), False)
])

empty_orders = spark.createDataFrame([], orders_schema)

table_path = f"s3a://{BUCKET}/bronze/orders"

(empty_orders
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("extract_date")
    .save(table_path))

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 60718)
Traceback (most recent call last):
  File "/usr/python/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/python/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/usr/python/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/python/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/home/airflow/.local/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/home/airflow/.local/lib/python3.11/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/home/airflow/.local/lib/python3.11/si

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, FloatType
)

# Shape of the raw extract file itself — no extract_date column here, that's
# added at Bronze-write time, not present in what the upstream system emits.
ORDERS_RAW_SCHEMA = StructType([
    StructField("order_id",     StringType(),    False),
    StructField("store_id",     StringType(),    False),
    StructField("rider_id",     StringType(),    True),
    StructField("status",       StringType(),    False),
    StructField("created_at",   TimestampType(), False),
    StructField("assigned_at",  TimestampType(), True),
    StructField("picked_up_at", TimestampType(), True),
    StructField("delivered_at", TimestampType(), True),
    StructField("order_total",  FloatType(),     False),
    StructField("updated_at",   TimestampType(), False),
])

RAW_BUCKET = "raw_incoming_data"


def read_orders_extract(spark: SparkSession, extract_date: str) -> DataFrame:
    """Reads one day's raw orders extract from the raw landing bucket.

    Args:
        spark: active SparkSession.
        extract_date: date the extract was taken, as "YYYY-MM-DD" — matches
            the suffix in the source filename, not necessarily today's date
            (a DAG re-run or backfill can request a past date).

    Returns:
        DataFrame matching ORDERS_RAW_SCHEMA, exactly as the file arrived.

    Raises:
        pyspark AnalysisException: if no file exists at that path. This is
        the real backstop for a late/missing upstream extract — the
        generator can legitimately not have run yet for `extract_date`.
    """
    file_path = f"s3a://{RAW_BUCKET}/orders/orders_{extract_date}.csv"

    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .schema(ORDERS_RAW_SCHEMA)
        .load(file_path)
    )

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit, to_date

BRONZE_BUCKET = "bronze-veloz"
BRONZE_ORDERS_PATH = f"s3a://{BRONZE_BUCKET}/orders"


def write_orders_bronze(df: DataFrame, extract_date: str) -> None:
    """Appends one day's raw orders extract to the Bronze Delta table.

    Args:
        df: raw extract DataFrame, matching ORDERS_RAW_SCHEMA — as returned
            by read_orders_extract(), untouched.
        extract_date: date the extract was taken, as "YYYY-MM-DD" — stamped
            onto every row and used as the partition value, so a rerun for
            the same date lands in the same partition rather than
            duplicating across two dates.
    """
    (
        df.withColumn("extract_date", to_date(lit(extract_date)))
        .write
        .format("delta")
        .mode("append")
        .partitionBy("extract_date")
        .save(BRONZE_ORDERS_PATH)
    )

In [ ]:
extract_date = "2026-08-30"

orders_df = read_orders_extract(spark, extract_date)
print(f"rows read from raw extract: {orders_df.count()}")

write_orders_bronze(orders_df, extract_date)

bronze_df = spark.read.format("delta").load(BRONZE_ORDERS_PATH)
bronze_df.filter(bronze_df.extract_date == extract_date).show(5)
print(f"rows now in bronze for {extract_date}: "
      f"{bronze_df.filter(bronze_df.extract_date == extract_date).count()}")